In [20]:
"""
    TP = 5
    TN = 0
    FP = 2
    FN = 3
"""
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
y_actual = [1,0,1,1,1,1,1,1,1,0]
y_pred   = [1,1,1,0,1,1,0,1,0,1]

print(confusion_matrix(y_actual, y_pred))
print("Accuracy:", accuracy_score(y_actual, y_pred))
print(classification_report(y_actual, y_pred))

[[0 2]
 [3 5]]
Accuracy: 0.5
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.71      0.62      0.67         8

    accuracy                           0.50        10
   macro avg       0.36      0.31      0.33        10
weighted avg       0.57      0.50      0.53        10



In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
df = pd.read_csv("../Data/bank-full.csv")
print("📊 데이터 shape:", df.shape)
print("타깃(y): yes/no → 이진 분류 → Logistic Regression")
df.head()

📊 데이터 shape: (45211, 17)
타깃(y): yes/no → 이진 분류 → Logistic Regression


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [22]:
print("결측치 before:", df.isnull().sum().sum())
def clean_missing_values(df):
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == 'object':
                df[col] = df[col].fillna(df[col].mode()[0])
            else:
                df[col] = df[col].fillna(df[col].median())
    return df
df = clean_missing_values(df)
print("결측치 after:", df.isnull().sum().sum())
df.info()

결측치 before: 0
결측치 after: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [23]:
# 3. Encoding (범주형 컬럼이 있으면 적용 - 이 데이터는 모두 수치형)
obj_cols = df.select_dtypes(include=['object']).columns
if len(obj_cols) > 0:
    for col in obj_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    print("Encoding 완료:", obj_cols.tolist())
else:
    print("범주형 컬럼 없음 - Encoding 스킵 (모두 수치형)")

Encoding 완료: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']


In [24]:
# 4. Scaling & X, y 분리
feature_cols = [c for c in df.columns if c != 'y']
X = df[feature_cols]
y = df['y']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
print("Scaling 완료 (StandardScaler)")

Scaling 완료 (StandardScaler)


In [25]:
# 5. Train/Test split & Logistic Regression 학습
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("✅ Logistic Regression 학습 완료")
print(f"Train Accuracy: {model.score(X_train, y_train):.4f}")
print(f"Test Accuracy:  {model.score(X_test, y_test):.4f}")

✅ Logistic Regression 학습 완료
Train Accuracy: 0.8918
Test Accuracy:  0.8879


In [26]:
# 6. Predict 결과
y_pred = model.predict(X_test)
result_df = pd.DataFrame({
    '실제값(y_test)': y_test.values,
    '예측값(y_pred)': y_pred,
    '정답여부': y_test.values == y_pred
})
result_df.index = y_test.index
print("📌 Predict 결과 (테스트셋 상위 20개):")
result_df.head(20)

📌 Predict 결과 (테스트셋 상위 20개):


,실제값(y_test),예측값(y_pred),정답여부
3776,0,0,True
9928,0,0,True
33409,0,0,True
31885,0,0,True
15738,0,0,True
30813,0,0,True
35463,0,0,True
31382,0,0,True
16904,0,0,True
11930,0,0,True


In [27]:
# 7. Metrics (각각 따로 출력)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support

print("Accuracy:   ", accuracy_score(y_test, y_pred))
print("Precision:  ", precision_score(y_test, y_pred, average='weighted'))
print("Recall:     ", recall_score(y_test, y_pred, average='weighted'))
print("F1-Score:   ", f1_score(y_test, y_pred, average='weighted'))

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred)
print("\n클래스별 Precision:", precision)
print("클래스별 Recall:   ", recall)
print("클래스별 F1-Score: ", f1)
print("클래스별 Support:  ", support)

Accuracy:    0.8878690699988941
Precision:   0.8645196751831744
Recall:      0.8878690699988941
F1-Score:    0.8640644818597165

클래스별 Support:
  클래스 0: 7952
  클래스 1: 1091
